# ANP: landing → Bronze Delta

**PREPARADO, NÃO EXECUTADO NO FABRIC.** Importe em um Notebook PySpark e anexe um Lakehouse de desenvolvimento como padrão. A Copy activity deve preservar o ZIP em Files/landing usando formato Binary. Este notebook não publica Silver nem Gold.

A entrega é uma versão completa de um semestre. Os bytes originais ficam preservados por hash. A Bronze mantém todas as linhas, inclusive duplicatas exatas, e pode conter várias versões do mesmo semestre; consumir todas juntas duplicaria observações.


In [ ]:
p_landing_zip = "/lakehouse/default/Files/landing/anp/2026-1/EXECUTION_ID/ca-2026-01.zip"
p_expected_zip_sha256 = "a2c95e5dfa324a9d7253d4e8d53022e0c08dcc5eb6f18551527ee9f29305c54c"
p_run_id = ""  # Identificador desta tentativa do notebook; vazio gera UUID.


In [ ]:
import csv
from datetime import datetime, timezone
import hashlib
import io
import json
from pathlib import Path
import re
import uuid
import zipfile
from pyspark.sql import functions as F, types as T

spark.conf.set("spark.sql.session.timeZone", "UTC")
run_id = p_run_id or str(uuid.uuid4())
if not re.fullmatch(r"[A-Za-z0-9_-]+", run_id):
    raise ValueError("run_id deve conter apenas letras, números, _ e -")
landing = Path(p_landing_zip).resolve()
files = Path("/lakehouse/default/Files").resolve()
if not landing.is_relative_to(files / "landing/anp"):
    raise ValueError("Entrada deve estar em Files/landing/anp no Lakehouse padrão")
if not re.fullmatch(r"[0-9a-f]{64}", p_expected_zip_sha256):
    raise ValueError("Informe o SHA-256 da versão aprovada para esta execução")
audit_dir = files / "audit/anp"
audit_dir.mkdir(parents=True, exist_ok=True)
audit_path = audit_dir / f"{run_id}.json"
if audit_path.exists():
    raise ValueError("Use um run_id novo por tentativa; a auditoria anterior é imutável")
audit = {"run_id": run_id, "status": "started", "environment": "fabric",
         "started_at_utc": datetime.now(timezone.utc).isoformat(),
         "landing_relative_path": landing.relative_to(files).as_posix(),
         "source_url": "https://www.gov.br/anp/pt-br/centrais-de-conteudo/dados-abertos/arquivos/shpc/dsas/ca/ca-2026-01.zip"}
audit_path.write_text(json.dumps(audit, indent=2), encoding="utf-8")


In [ ]:
headers = ["Regiao - Sigla", "Estado - Sigla", "Municipio", "Revenda", "CNPJ da Revenda",
           "Nome da Rua", "Numero Rua", "Complemento", "Bairro", "Cep", "Produto",
           "Data da Coleta", "Valor de Venda", "Valor de Compra", "Unidade de Medida", "Bandeira"]
columns = ["regiao_sigla", "uf", "municipio", "revenda", "cnpj_revenda", "nome_rua",
           "numero_rua", "complemento", "bairro", "cep", "produto", "data_coleta",
           "valor_venda", "valor_compra", "unidade_medida", "bandeira"]

try:
    payload = landing.read_bytes()
    sha = hashlib.sha256(payload).hexdigest()
    audit.update({"source_sha256": sha, "expected_source_sha256": p_expected_zip_sha256})
    if sha != p_expected_zip_sha256:
        raise ValueError("Hash inesperado: investigar republicação ou cópia incompleta")
    raw_dir = files / "raw/anp_2026_1" / sha
    raw_dir.mkdir(parents=True, exist_ok=True)
    preserved = raw_dir / "ca-2026-01.zip"
    if preserved.exists():
        if hashlib.sha256(preserved.read_bytes()).hexdigest() != sha:
            raise ValueError("ZIP preservado foi alterado")
    else:
        preserved.write_bytes(payload)
    with zipfile.ZipFile(io.BytesIO(payload)) as archive:
        members = [m for m in archive.infolist() if not m.is_dir()]
        if len(members) != 1 or not members[0].filename.lower().endswith(".csv"):
            raise ValueError("Esperado ZIP com um CSV")
        content = archive.read(members[0])
        member_name = members[0].filename
    parsed = csv.reader(io.StringIO(content.decode("utf-8-sig")), delimiter=";", strict=True)
    if next(parsed) != headers:
        raise ValueError("Layout mudou: revisar contrato")
    row_count = 0
    for row in parsed:
        if len(row) != len(headers):
            raise ValueError("CSV com quantidade inválida de campos")
        row_count += 1
    if not row_count:
        raise ValueError("Entrega vazia")
    csv_sha = hashlib.sha256(content).hexdigest()
    extracted = files / "extracted/anp_2026_1" / sha / "source.csv"
    extracted.parent.mkdir(parents=True, exist_ok=True)
    if extracted.exists() and hashlib.sha256(extracted.read_bytes()).hexdigest() != csv_sha:
        raise ValueError("CSV extraído foi alterado")
    if not extracted.exists():
        extracted.write_bytes(content)
    audit.update({"source_sha256": sha, "csv_sha256": csv_sha, "csv_member": member_name,
                  "raw_bytes": len(payload), "source_rows": row_count})
    schema = T.StructType([T.StructField(name, T.StringType(), True) for name in headers])
    raw = (spark.read.schema(schema).option("header", True).option("sep", ";")
           .option("encoding", "UTF-8").option("mode", "FAILFAST")
           .option("multiLine", True).option("quote", '"').option("escape", '"')
           .csv(extracted.relative_to(files.parent).as_posix()).toDF(*columns).fillna(""))
    raw = raw.cache()
    if raw.count() != row_count:
        raise ValueError("Contagem Spark difere do parser CSV")
    bronze = (raw.withColumn("raw_row_sha256", F.sha2(F.to_json(F.struct(*[F.col(c) for c in columns])), 256))
              .withColumn("source_id", F.lit("anp_2026_1"))
              .withColumn("source_sha256", F.lit(sha))
              .withColumn("csv_sha256", F.lit(csv_sha))
              .withColumn("run_id", F.lit(run_id))
              .withColumn("ingested_at_utc", F.lit(audit["started_at_utc"])))
    # Execuções seriais: substitui apenas a MESMA versão; outras versões permanecem.
    # Suporte de replaceWhere em colunas de dados deve ser confirmado no runtime do Fabric.
    (bronze.write.format("delta").mode("overwrite")
     .option("replaceWhere", f"source_sha256 = '{sha}'").saveAsTable("bronze_anp"))
    persisted = spark.table("bronze_anp").where(F.col("source_sha256") == sha).count()
    if persisted != row_count:
        raise ValueError("Reconciliação Bronze falhou; não avançar consumo")
    audit.update({"status": "bronze_validated", "bronze_rows_for_version": persisted,
                  "finished_at_utc": datetime.now(timezone.utc).isoformat()})
except Exception as exc:
    audit.update({"status": "failed", "error_type": type(exc).__name__,
                  "finished_at_utc": datetime.now(timezone.utc).isoformat()})
    raise
finally:
    audit_path.write_text(json.dumps(audit, ensure_ascii=False, indent=2), encoding="utf-8")
    if "raw" in globals():
        raw.unpersist()
print(json.dumps(audit, ensure_ascii=False))


## Evidência necessária antes de chamar esta etapa de validada

Executar no Fabric, registrar runtime, Lakehouse, run_id e resultado da Copy activity sem segredos. Exportar a auditoria de Files/audit/anp e conferir 422.418 linhas para o hash inicial. Repetir com run_id novo e verificar a mesma contagem por source_sha256. O hash de linha ajuda a localizar duplicatas exatas; ele não é a chave de negócio da coleta.

Não há marcador de versão ativa nem transação entre Bronze/Silver/Gold neste notebook inicial. Uma republicação deve ser preservada como outra versão e passar pelas validações da etapa seguinte antes de substituir o escopo de consumo. Nenhuma Silver deve concatenar automaticamente todas as versões da Bronze.
